# 🏠 Housing Price Prediction — Ordinary Linear Regression
This notebook trains a **Linear Regression** model on the California Housing dataset.  
Upload your `housing.csv` from your local machine using the file picker below.

In [ ]:
# ── Install dependencies (Colab already has most, just in case) ──────────────
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# ── Upload dataset from local machine ────────────────────────────────────────
from google.colab import files

print('📂 Please select your housing.csv file...')
uploaded = files.upload()          # opens the file picker
filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {filename}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import io
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✅')

In [ ]:
# ── Load & preview data ───────────────────────────────────────────────────────
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Basic info & missing values ───────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
# Drop rows with missing values (total_bedrooms has 207 nulls)
df = df.dropna()

# Encode categorical column 'ocean_proximity' with LabelEncoder
le = LabelEncoder()
df['ocean_proximity_enc'] = le.fit_transform(df['ocean_proximity'])

# Features and target
feature_cols = [
    'longitude', 'latitude', 'housing_median_age',
    'total_rooms', 'total_bedrooms', 'population',
    'households', 'median_income', 'ocean_proximity_enc'
]
X = df[feature_cols]
y = df['median_house_value']

print('Features:', feature_cols)
print('Target: median_house_value')

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

In [ ]:
# ── Train Linear Regression ───────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'MSE  : {mse:,.2f}')
print(f'RMSE : {rmse:,.2f}')
print(f'R²   : {r2:.4f}')

## 📊 Visualizations

In [ ]:
# ── 1. Actual vs Predicted ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.3, edgecolors='k', linewidths=0.4, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual House Value ($)', fontsize=12)
ax.set_ylabel('Predicted House Value ($)', fontsize=12)
ax.set_title('Linear Regression — Actual vs Predicted', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Residuals Distribution ────────────────────────────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Residuals Distribution', fontsize=13)
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')

# Residuals vs Fitted
axes[1].scatter(y_pred, residuals, alpha=0.3, color='steelblue', edgecolors='k', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Fitted Values', fontsize=13)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Feature Coefficients ───────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['Coefficient']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression — Feature Coefficients', fontsize=14)
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Correlation Heatmap ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[feature_cols + ['median_house_value']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. House Value by Ocean Proximity (box plot) ─────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column='median_house_value', by='ocean_proximity', ax=ax,
           patch_artist=True, vert=True)
ax.set_title('House Value by Ocean Proximity', fontsize=13)
ax.set_xlabel('Ocean Proximity')
ax.set_ylabel('Median House Value ($)')
plt.suptitle('')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(model, 'housing_linear_regression_model.joblib')
print('Model saved as housing_linear_regression_model.joblib')

# Download to local machine
files.download('housing_linear_regression_model.joblib')